# Notebook 2: Run Hardware Inference Benchmark

This notebook runs the fixed BFCL subset on one hardware backend.

It records:
- raw model output
- latency
- token counts
- tokens/sec
- GPU memory usage
- errors

The same notebook is used on each accelerator by changing the config file.

In [1]:
import json
import time
from pathlib import Path
from datetime import datetime

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/Users/joe/coding/CSE188/BFCL-hardware-efficiency-study/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path("../data")
CONFIG_DIR = Path("../configs")
RESULTS_DIR = Path("../results/raw")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TASK_PATH = DATA_DIR / "bfcl_subset.json"

# Change this depending on hardware:
CONFIG_PATH = CONFIG_DIR / "t4_config.json"
# CONFIG_PATH = CONFIG_DIR / "l4_config.json"
# CONFIG_PATH = CONFIG_DIR / "rtx6000_config.json"

In [3]:
def load_config(path):
    with open(path, "r", encoding="utf-8") as f:
        config = json.load(f)

    if "inherits" in config:
        parent_path = CONFIG_DIR / config["inherits"]
        with open(parent_path, "r", encoding="utf-8") as f:
            parent = json.load(f)

        parent.update({k: v for k, v in config.items() if k != "inherits"})
        return parent

    return config

config = load_config(CONFIG_PATH)
config

{'model_name': 'Qwen/Qwen2.5-7B-Instruct',
 'temperature': 0.0,
 'max_new_tokens': 256,
 'batch_size': 1,
 'num_repeats': 1,
 'device': 'cuda',
 'dtype': 'auto',
 'backend': 'transformers',
 'hardware': 'T4'}

In [4]:
with open(TASK_PATH, "r", encoding="utf-8") as f:
    tasks = json.load(f)

print(f"Loaded {len(tasks)} BFCL tasks.")
print("Categories:", pd.Series([t["category"] for t in tasks]).value_counts().to_dict())
print("First task ID:", tasks[0]["task_id"])

Loaded 100 BFCL tasks.
Categories: {'simple': 25, 'multiple': 25, 'parallel': 25, 'parallel_multiple': 25}
First task ID: simple_361


## Hardware Check

In [5]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
    print("Total memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("No CUDA GPU detected. This notebook should be run on GPU hardware for benchmark runs.")

CUDA available: False
No CUDA GPU detected. This notebook should be run on GPU hardware for benchmark runs.


## Load Model and Tokenizer

In [6]:
model_name = config["model_name"]

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded:", model_name)

Tokenizer loaded: Qwen/Qwen2.5-7B-Instruct


In [7]:
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("Model loaded:", model_name)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 4 files:   0%|          | 0/4 [06:06<?, ?it/s]


KeyboardInterrupt: 

## Prompt Formatting

BFCL prompts are stored in chat-message format. Each task also includes function/tool schemas. We use the model tokenizer's chat template where possible.

In [ ]:
def normalize_messages(prompt):
    """
    BFCL stores question as nested chat turns, often like:
    [[{"role": "user", "content": "..."}]]

    This function flattens it into:
    [{"role": "user", "content": "..."}]
    """
    if isinstance(prompt, list) and len(prompt) > 0:
        if isinstance(prompt[0], list):
            return prompt[0]
        if isinstance(prompt[0], dict):
            return prompt

    return [{"role": "user", "content": str(prompt)}]


def format_prompt(task):
    messages = normalize_messages(task["prompt"])
    tools = task["functions_or_tools"]

    try:
        return tokenizer.apply_chat_template(
            messages,
            tools=tools,
            tokenize=False,
            add_generation_prompt=True
        )
    except Exception:
        # Fallback if tool template is unsupported
        return (
            "You are a tool-calling assistant.\n\n"
            f"Available tools:\n{json.dumps(tools, indent=2)}\n\n"
            f"User request:\n{messages[-1]['content']}\n\n"
            "Return the correct function call."
        )

In [ ]:
sample_prompt = format_prompt(tasks[0])
print(sample_prompt[:3000])

## Inference Function

This function runs one task, measures latency, counts tokens, and records GPU memory.

In [ ]:
def run_one_task(task, run_index=0):
    prompt_text = format_prompt(task)

    result = {
        "task_id": task["task_id"],
        "category": task["category"],
        "hardware": config["hardware"],
        "model": config["model_name"],
        "backend": config.get("backend", "transformers"),
        "run_index": run_index,
        "raw_output": None,
        "latency_s": None,
        "input_tokens": None,
        "output_tokens": None,
        "tokens_per_second": None,
        "peak_memory_mb": None,
        "error": None,
        "timestamp": datetime.utcnow().isoformat()
    }

    try:
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
        input_tokens = inputs["input_ids"].shape[-1]

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()

        start = time.perf_counter()

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=config["max_new_tokens"],
                do_sample=False,
                temperature=None,
                pad_token_id=tokenizer.eos_token_id
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        latency_s = time.perf_counter() - start

        generated_ids = output_ids[0][input_tokens:]
        output_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

        output_tokens = generated_ids.shape[-1]
        tokens_per_second = output_tokens / latency_s if latency_s > 0 else None

        peak_memory_mb = None
        if torch.cuda.is_available():
            peak_memory_mb = torch.cuda.max_memory_allocated() / 1024**2

        result.update({
            "raw_output": output_text,
            "latency_s": latency_s,
            "input_tokens": int(input_tokens),
            "output_tokens": int(output_tokens),
            "tokens_per_second": tokens_per_second,
            "peak_memory_mb": peak_memory_mb
        })

    except Exception as e:
        result["error"] = str(e)

    return result

## Test Run on Two Tasks

Before running the full benchmark, test a small subset.

In [ ]:
test_results = []

for task in tasks[:2]:
    result = run_one_task(task, run_index=0)
    test_results.append(result)
    print("\nTASK:", result["task_id"], result["category"])
    print("ERROR:", result["error"])
    print("LATENCY:", result["latency_s"])
    print("OUTPUT:\n", result["raw_output"][:1000] if result["raw_output"] else None)

pd.DataFrame(test_results)

## Full Benchmark Run

This writes results incrementally as JSONL so partial runs are recoverable.

In [ ]:
hardware = config["hardware"]
output_path = RESULTS_DIR / f"{hardware}_results.jsonl"

print("Output path:", output_path)

In [ ]:
RUN_FULL_BENCHMARK = False  # Change to True when ready

if RUN_FULL_BENCHMARK:
    with open(output_path, "a", encoding="utf-8") as f:
        for i, task in enumerate(tasks):
            print(f"[{i+1}/{len(tasks)}] Running {task['task_id']} ({task['category']})")

            result = run_one_task(task, run_index=0)

            f.write(json.dumps(result) + "\n")
            f.flush()

            print("  error:", result["error"])
            print("  latency:", result["latency_s"])
            print("  toks/sec:", result["tokens_per_second"])

    print("Benchmark complete.")
else:
    print("RUN_FULL_BENCHMARK is False. Set it to True to run all tasks.")

## Load Saved Results

In [ ]:
def load_jsonl(path):
    rows = []
    if not path.exists():
        return rows

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

    return rows

saved_rows = load_jsonl(output_path)
df_results = pd.DataFrame(saved_rows)

print("Saved rows:", len(df_results))
df_results.head()

In [ ]:
if len(df_results) > 0:
    display(df_results.groupby("category")[["latency_s", "tokens_per_second", "peak_memory_mb"]].mean())
    display(df_results[["latency_s", "tokens_per_second", "input_tokens", "output_tokens", "peak_memory_mb"]].describe())
else:
    print("No saved rows yet.")